# Correlation Leakage - Nov 10 cluster visualization


In [1]:
from pathlib import Path
# ^^^ pyforest auto-imports - don't write above this line
%load_ext autoreload
%autoreload 2

In [15]:
import random
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
from networkx.algorithms.community import modularity
import networkx as nx
import numpy as np
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.figure_factory as ff
from ipywidgets import widgets

import sys
sys.path.extend(['./', '../'])

from src.clustering import LFRGraph, CorrelationClustering, MarkovClustering, OtherClustering
from src.utils import Paths
from cdlib import NodeClustering

In [12]:
import sys
sys.path.extend(['./', '../'])

from src.clustering import LFRGraph, CorrelationClustering, MarkovClustering, OtherClustering
from src.utils import Paths
from cdlib import NodeClustering

Note: to be able to use all overlapping methods, you need to install some additional packages:  {'ASLPAw'}


In [19]:
%matplotlib inline

# plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 20
plt.rcParams['axes.labelsize'] = 20
plt.rcParams['axes.titlesize'] = 20

pd.options.plotting.backend = 'plotly'
pio.templates.default = 'presentation+plotly_dark'
pio.renderers.default = 'iframe'

sns.set_context('talk')
sns.set_style("ticks")

SMALL_SIZE = 30
MEDIUM_SIZE = 30
BIGGER_SIZE = 30

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=20)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [4]:
def varying_tau_plot(metric, methods):
    colors = {method: px.colors.qualitative.Pastel[i] for i, method in enumerate(methods)}
    fill_alpha = 0.35
    fill_colors = {method: (c[: c.rfind(')')] + f', {fill_alpha})').replace('rgb', 'rgba')  for method, c in colors.items()}
    fig = go.Figure()

    df = (
        pd.concat([cl_results_df, mc_results_df], ignore_index=True)
        .rename(columns={'num_clusters': 'clusters'})
        .query('method.isin(@methods)')
    )

    # Add traces, multiple plots for each slider step
    for tau_ix in range(df.tau_ix.max() + 1):
        for i, method in enumerate(methods):  # for each method
            filt_df = (
                df
                .query('tau_ix==@tau_ix & method==@method', engine='python')
            )

            if len(filt_df) == 0:
                fig.add_trace(go.Scatter(name=f'{method}_{tau_ix}_blank', visible=False))
                fig.add_trace(go.Scatter(name=f'{method}_{tau_ix}_blank_error', visible=False))
            else:
                df_ = (
                    filt_df
                    .pivot_table(
                    index=['method', 'mu'], 
                    aggfunc=['mean', 'sem'],
                    fill_value=0)
                    .reset_index()
                )
                df_.columns = df_.columns.to_flat_index().str.join('_') 
                df_ = df_.rename(columns={'mu_': 'mu', 'method_': 'method'})

                x = df_.mu.values.tolist()
                y = df_[f'mean_{metric}'].tolist()
                y_upper = (df_[f'mean_{metric}'] + df_[f'sem_{metric}']).tolist()
                y_lower = (df_[f'mean_{metric}'] - df_[f'sem_{metric}']).tolist()

                # error bar
                fig.add_trace(
                    go.Scatter(
                        x=x+x[::-1], # x, then x reversed
                        y=y_upper+y_lower[::-1], # upper, then lower reversed
                        fill='toself',
                        fillcolor=fill_colors[method],
                        line=dict(color='rgba(255,255,255,0)', width=0),
                        hoverinfo="skip",
                        showlegend=False,
                        visible=False,
                        legendgroup=method,
                    )
                )

                # actual data 
                fig.add_trace(
                    go.Scatter(
                        visible=False,
                        marker_color=colors[method],
                        legendgroup=method,
                        name=method,
                        x=df_['mu'],
                        y=df_[f'mean_{metric}'],
                        mode='markers+lines',
                        opacity=0.7)
                )


    # Make 0th trace visible
    for i in range(2*len(methods)):
        fig.data[i].visible = True

    # Create and add slider
    steps = []
    # title_strs = ['tau = i' for i in ]
    step_labels = [''] * df.tau_ix.nunique()
    step_labels[0] = '0'
    step_labels[4] = 'max'
    step_labels[-1] = 'diff'

    for i in range(0, len(fig.data), 2*len(methods)):  # skip over lines for the same method
        i_ = i // (2 * len(methods))
        step = dict(
            method="update",
            label=step_labels[i_],
            args=[{"visible": [False] * len(fig.data)},
                  # {"title": f'{step_labels[i_]}'},
                 ],  # layout attribute
        )
        for j in range(0, 2*len(methods), 2):  # turn on i and i + j 
            step["args"][0]["visible"][i+j] = True  # Toggle i'th trace to "visible"
            step["args"][0]["visible"][i+j+1] = True  # Toggle i'th trace to "visible"
        steps.append(step)


    sliders = [dict(
        active=0,
        name='ix',
        currentvalue={"prefix": r"Tau: "},
        pad={"t": 50},
        steps=steps
    )]

    fig.update_layout(
        title=f'Variation of {metric!r} over taus for {", ".join(list(methods))!r}',
        sliders=sliders
    )
    fig.update_xaxes(title=r'$\mu$')

    # y_range = (0, df.clusters.max()) if metric == 'clusters' else (0, 1)
    y_range = None if metric == 'clusters' else (-0.05, 1.05)
    fig.update_yaxes(range=y_range, title=metric)

    return fig

In [5]:
def _make_plot(df, suptitle, cmap='tab10'):
    hue_order = sorted(df.method.unique())
    ncol = len(hue_order) // 9 + 1
    if ncol > 1:
        figsize = (25, 10)
    else:
        figsize = (15, 10)
    fig, axs = plt.subplots(figsize=figsize, ncols=2, nrows=2, sharex=True, layout='constrained')

    axs = axs.flatten()

    pal = sns.color_palette(cmap, len(hue_order))
    for i, (ax, y) in enumerate(zip(axs, ['num_clusters', 'modularity', 'NMI', 'EC'])):
        sns.lineplot(data=df, x='mu', y=y, hue='method', hue_order=hue_order, marker='o', ax=ax, palette=pal, alpha=0.75);

        if i == 0:  # add baseline for before and num_clusters
            sns.lineplot(data=df, x='mu', y='orig_clusters', marker='o', color='black', ax=ax)

        ax.grid();
        # Hide the right and top spines
        ax.spines.right.set_visible(False)
        ax.spines.top.set_visible(False)

        # Only show ticks on the left and bottom spines
        ax.yaxis.set_ticks_position('left');
        ax.xaxis.set_ticks_position('bottom');

        if i == 1:
            # get the legend object
            leg = ax.legend(bbox_to_anchor=(1.04, 0.25), loc="lower left", borderaxespad=0, ncol=ncol)
            # change the line width for the legend
            for line in leg.get_lines():
                line.set_linewidth(6.0)

        else:
            ax.legend().set_visible(False)

    fig.suptitle(suptitle);
    return 

# data prep

In [13]:
paths = Paths()
cl_results_path = paths.results_dir / 'stats_CL.csv'
mc_results_path = paths.results_dir / 'stats_MC.csv'
other_results_path = paths.results_dir / 'stats_others.csv'

In [7]:
cl_results_df = pd.read_csv(cl_results_path, on_bad_lines='skip')

cl_results_df = (
    cl_results_df
    .assign(method=lambda df_: df_.method.str.replace('-exact', '', regex=False))
)
mask = ~((cl_results_df.method=='GW-C') & (cl_results_df.tau_ix>4))  # delete rows with tau > tau_max for GW 
cl_results_df = cl_results_df[mask]
cl_results_df.drop_duplicates(subset=cl_results_df.columns[1: ], inplace=True, keep='last')



display(len(cl_results_df), f'{cl_results_df.method.unique().tolist()}', cl_results_df.tail(2))

109791

"['GW-C', 'RW-D', 'RW-C', 'Diff-C', 'GW-D', 'CO-C', 'Diff-C-ER', 'RW-C-ER', 'CO-C-ER', 'RW-D-ER']"

,timestamp,name,mu,avg_k,max_k,method,tau_max,tau_diff,tau_ix,tau,orig_clusters,running_time,metric,score,num_clusters,modularity,NMI,AMI,EC,output_cluster_path
119869,Thu Nov 10 23:27:24 2022,SBM-big-small-0.1,NaN,NaN,NaN,RW-C,0.8597,3.467957,7,2.815894,4,0.1255,leakage_score,21.153674,5,0.2956,0.6507,0.6412,0.4930,/N/slate/ssikdar/data-dump/corr-cluster/result...
119870,Thu Nov 10 23:27:25 2022,SBM-big-small-0.1,NaN,NaN,NaN,RW-C,0.8597,3.467957,8,3.467957,4,0.1706,leakage_score,14.286019,66,0.1705,0.4897,0.3260,0.3212,/N/slate/ssikdar/data-dump/corr-cluster/result...


In [8]:
mc_results_df = pd.read_csv(mc_results_path)
mc_results_df.drop_duplicates(subset=mc_results_df.columns[1: ], inplace=True, keep='last')

display(len(mc_results_df), mc_results_df.method.unique(), mc_results_df.tail(2))

17412

array(['MC-Diff-C', 'MC-RW-C'], dtype=object)

,timestamp,name,mu,avg_k,max_k,method,tau_max,tau_diff,tau_ix,tau,orig_clusters,metric,score,running_time,num_clusters,modularity,NMI,AMI,EC,output_cluster_path
17410,Thu Nov 10 23:27:38 2022,SBM-big-small-0.1,NaN,NaN,NaN,MC-Diff-C,0.009,0.054086,7,0.042804,4,0.1035,leakage_score,0.280358,4,0.3241,0.8,0.7964,0.625,/N/slate/ssikdar/data-dump/corr-cluster/result...
17411,Thu Nov 10 23:27:39 2022,SBM-big-small-0.1,NaN,NaN,NaN,MC-Diff-C,0.009,0.054086,8,0.054086,4,0.1231,leakage_score,0.224743,4,0.3241,0.8,0.7964,0.625,/N/slate/ssikdar/data-dump/corr-cluster/result...


In [9]:
other_results_df = pd.read_csv(other_results_path)
other_results_df.drop_duplicates(subset=other_results_df.columns[1: ], inplace=True, keep='last')

other_results_df = (
    other_results_df
    .query('method!="method"')
    .convert_dtypes(convert_floating=True, convert_integer=True)
    .astype({'num_clusters': 'int'})
    .assign(method=lambda df_: df_.method.replace({'greedy_modularity': 'CNM',
                                                   'label_propagation': 'Label Prop',
                                                   'infomap': 'Infomap',
                                                   'eigenvector': 'eig',
                                                   'sbm_dl': 'SBM',
                                                   'sbm_dl_nested': 'Nested SBM'}))
)

display(len(other_results_df), f'{other_results_df.method.unique().tolist()}', other_results_df.tail(2))

9432

"['leiden', 'Label Prop', 'Infomap', 'louvain', 'CNM', 'SBM', 'Nested SBM']"

,timestamp,name,mu,avg_k,max_k,method,orig_clusters,metric,score,running_time,num_clusters,modularity,NMI,AMI,EC,output_cluster_path
9430,Wed 02 Nov 2022 08:21:58 PM,LFR-1k-50-20-75_5,0.75,20,50,louvain,53,modularity,0.228106,0.3684,14,0.2281,0.4924,0.4237,0.1444,/N/slate/ssikdar/data-dump/corr-cluster/result...
9431,Wed 02 Nov 2022 08:21:55 PM,LFR-1k-50-15-65_12,0.65,15,50,louvain,53,modularity,0.330933,0.3376,20,0.3309,0.8327,0.8011,0.4276,/N/slate/ssikdar/data-dump/corr-cluster/result...


# Run clustering on new graphs

* Try more SBMs?
* Ring of cliques? For clique size C, community label of node i is i // C. 
* Core periphery networks w/ SBMs - Ryan Gallagher's paper

In [3]:
def dict_to_list_of_lists(d):
    new_d = {}
    for key, val in d.items():
        if val not in new_d:
            new_d[val] = []
        new_d[val].append(key)
    return list(new_d.values())

def read_file_into_dict(path):
    d = {}
    with open(path) as reader:
        for line in reader:
            x, y = map(int, line.split())
            d[x] = y
    return d

In [17]:
def construct_graph(name):
    if name == 'SBM-asymm':
        sizes = [30, 30, 30]
        probs = [
           [0.35, 0.05, 0.05],
           [0.05, 0.35, 0.1],
           [0.05, 0.1, 0.35]
         ]
    elif name == 'SBM-symm':
        sizes = [30, 30, 30]
        probs = [
           [0.35, 0.05, 0.05],
           [0.05, 0.35, 0.05],
           [0.05, 0.05, 0.35]
         ]
    elif name == 'SBM-easy':
        sizes = [50, 50, 50, 50]
        mu = 0.1 
        p_in, p_out = 1 - mu, mu
        probs = np.ones((len(sizes), len(sizes))) * p_out
        np.fill_diagonal(probs, p_in)
    elif name == 'SBM-big-small':
        sizes = [100, 50, 25, 25]
        mu = 0.1 
        p_in, p_out = 1 - mu, mu
        probs = np.ones((len(sizes), len(sizes))) * p_out
        np.fill_diagonal(probs, p_in)
    else:
        raise NotImplementedError()
        
    G = nx.stochastic_block_model(sizes=sizes, p=probs, seed=1)
    G.name = name
    nx.set_node_attributes(G, name='pos', values=nx.spring_layout(G))

    ground_truth_comm_labels = nx.get_node_attributes(G, name='block')
    ground_truth_clustering = NodeClustering(dict_to_list_of_lists(ground_truth_comm_labels), graph=G)

    return G, ground_truth_comm_labels, ground_truth_clustering

In [5]:
def get_edge_trace(G, edge_width, inter_edge_color, intra_edge_color, comm_labels):
    if comm_labels is None:
        comm_labels = {}
        
    inter_edge_width = edge_width  
    intra_edge_width = edge_width * 1.5

    intra_edge_x, intra_edge_y = [], []
    inter_edge_x, inter_edge_y = [], []
    
    for u, v in G.edges():
        x0, y0 = G.nodes[u]['pos']
        x1, y1 = G.nodes[v]['pos']
        
        if comm_labels.get(u, u) == comm_labels.get(v, v):
            intra_edge_x.extend([x0, x1, None])
            intra_edge_y.extend([y0, y1, None])
        else:
            inter_edge_x.extend([x0, x1, None])
            inter_edge_y.extend([y0, y1, None])

    intra_edge_trace = go.Scattergl(
        x=intra_edge_x, y=intra_edge_y,
        line=dict(width=intra_edge_width, color=intra_edge_color),
        hoverinfo='skip',
        opacity=0.8,
        mode='lines')
            
    inter_edge_trace = go.Scattergl(
        x=inter_edge_x, y=inter_edge_y,
        line=dict(width=inter_edge_width, color=inter_edge_color),
        hoverinfo='skip',
        opacity=0.8,
        mode='lines')
    return intra_edge_trace, inter_edge_trace

def get_node_trace(G, node_size, marker_width):
    node_x, node_y = [], []

    for node in G.nodes():
        x, y = G.nodes[node]['pos']
        node_x.append(x)
        node_y.append(y)

    node_trace = go.Scattergl(
        name='nodes',
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        marker=dict(
            showscale=False,
            color=[],
            size=node_size,
            line=dict(
                width=marker_width,
                color='black',
            ),
            line_width=2)
    )

    return node_trace


def update_node_colors_and_texts(G, comm_labels):
    """
    Return updated node trace of node colors and texts based on comm_labels 
    """
    node_colors, node_text = [], []
    # colors = px.colors.qualitative.Pastel
    colors = px.colors.qualitative.Plotly
    
    if comm_labels is not None: 
        consecutive_comm_label_map = {k: i for i, k in enumerate(set(comm_labels.values()))}
        num_clusters = len(set(comm_labels.values()))

        for node in G.nodes():
            comm_label = consecutive_comm_label_map[comm_labels[node]]
            if comm_label > len(colors) - 1:
                color = 'silver'
            else:
                color = colors[comm_label]
            node_colors.append(color)
            node_text.append(f'Node: {node}<br>Cluster: {comm_label}')
    else:
        node_colors = ['silver'] * G.order()
        node_text = [f'Node: {node}' for node in G.nodes()]

    return node_colors, node_text


def plot_graph(G, comm_labels=None, layout=None, node_size=12, edge_width=0.20, edge_color='gray', marker_width=10, fig=None, return_traces=False):
    if G.order() < 50:
        node_size = 20
        edge_width = 1.5
        edge_color = 'silver'
        
    if fig is None:
        fig = go.Figure()
    
    if layout is not None:
        layout_alg = getattr(nx.layout, f'{layout}_layout')
        nx.set_node_attributes(G, name='pos', values=layout_alg(G))
    
    intra_edge_trace, inter_edge_trace = get_edge_trace(G, edge_width=edge_width, inter_edge_color=edge_color, intra_edge_color='silver',
                                                        comm_labels=comm_labels)
    node_trace = get_node_trace(G, marker_width=marker_width, node_size=node_size)
    
    node_colors, node_text = update_node_colors_and_texts(G=G, comm_labels=comm_labels)
    node_trace.marker.color = node_colors
    node_trace.text = node_text
    
    traces =  intra_edge_trace, inter_edge_trace, node_trace  # nodes drawn at the end
    fig.add_traces(traces)
    
    fig.update_layout(showlegend=False, 
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
    if return_traces:
        return traces
    else:
        return fig
    

In [6]:
def make_community_heatmap(G, comm_labels):
    nx.set_node_attributes(G, name='cluster', values=comm_labels)
    mat = nx.attribute_mixing_matrix(G, attribute='cluster')
    # iu = np.triu_indices(mat.shape[0], 1)
    # mat[iu] = None
    text = np.round(mat, 3)
    
    clu_text = [f'{i}' for i in range(mat.shape[0])]
    heatmap = go.Heatmap(x=clu_text, y=clu_text, z=mat, text=text, texttemplate='%{text}',
                         showscale=False, hoverongaps=False,
                        )
    return heatmap

In [8]:
def plot_graph_and_heatmap(G, ground_truth_comm_labels):
    fig = go.Figure()

    fig = make_subplots(
            rows=1, cols=2, subplot_titles=['Ground truth clustering', 'Heatmap'],
            column_widths=[0.7, 0.3],
            vertical_spacing=0.0,
        )


    traces = plot_graph(G, comm_labels=ground_truth_comm_labels, node_size=15, edge_width=0.5, return_traces=True)
    fig.add_traces(traces, rows=1, cols=1)
    # fig.add_trace(node_trace, row=1, col=1)

    fig.add_trace(make_community_heatmap(G, comm_labels=ground_truth_comm_labels), row=1, col=2)
    fig.update_yaxes(autorange='reversed', row=1, col=2)

    fig.update_layout(showlegend=False, 
                      xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                      yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                      height=700,
                      title=f'{G.name}    |V| = {G.order():,}   |E| = {G.size():,}')


    return fig

In [20]:
name = 'SBM-easy'
G, ground_truth_comm_labels, ground_truth_clustering = construct_graph(name=name)
plot_graph_and_heatmap(G, ground_truth_comm_labels)

In [ ]:
def run_clustering_method(G, ground_truth_clustering, method):
    if method in ['Diff', 'CO', 'RW']:  # correlation leakage
        for er_null_model in [True, False]:
            family = 'toy'
            corr_cluster = CorrelationClustering(paths=paths, input_graph=G, continuous=True, method=method, family=family,
                                                 ground_truth_clustering=ground_truth_clustering, erdos_renyi_null=er_null_model)
            clusterings, best_clusterings = corr_cluster.run(num_trials=5, overwrite=True)
    elif 'MC' in method:
        

In [157]:
# correlation leakage methods
# method = 'Diff'
for method in ['Diff', 'CO', 'RW']:
    for er_null_model in [True, False]:
        family = 'toy'
        corr_cluster = CorrelationClustering(paths=paths, input_graph=G, continuous=True, method=method, family=family,
                                             ground_truth_clustering=ground_truth_clustering, erdos_renyi_null=er_null_model)

        clusterings, best_clusterings = corr_cluster.run(num_trials=5, overwrite=True)

CorrelationClustering Diff-C-ER mu=nan tau_ix=0 ..:   0%|          | 0/5 [00:00<?, ?it/s]

CorrelationClustering Diff-C-ER mu=nan tau_ix=1 ..:   0%|          | 0/5 [00:00<?, ?it/s]

CorrelationClustering Diff-C-ER mu=nan tau_ix=2 ..:   0%|          | 0/5 [00:00<?, ?it/s]

CorrelationClustering Diff-C-ER mu=nan tau_ix=3 ..:   0%|          | 0/5 [00:00<?, ?it/s]

CorrelationClustering Diff-C-ER mu=nan tau_ix=4 ..:   0%|          | 0/5 [00:00<?, ?it/s]

...

CorrelationClustering RW-C mu=nan tau_ix=4 ..:   0%|          | 0/5 [00:00<?, ?it/s]

CorrelationClustering RW-C mu=nan tau_ix=5 ..:   0%|          | 0/5 [00:00<?, ?it/s]

CorrelationClustering RW-C mu=nan tau_ix=6 ..:   0%|          | 0/5 [00:00<?, ?it/s]

CorrelationClustering RW-C mu=nan tau_ix=7 ..:   0%|          | 0/5 [00:00<?, ?it/s]

CorrelationClustering RW-C mu=nan tau_ix=8 ..:   0%|          | 0/5 [00:00<?, ?it/s]

In [158]:
# correlation leakage methods
# method = 'Diff'
for method in ['Diff']:
    family = 'toy'
    markov_cluster = MarkovClustering(paths=paths, input_graph=G, continuous=True, method=method, family=family,
                                         ground_truth_clustering=ground_truth_clustering)

    clusterings, best_clusterings = markov_cluster.run(num_trials=10, overwrite=True)

MarkovClustering MC-Diff-C mu=nan tau_ix=0 ..:   0%|          | 0/10 [00:00<?, ?it/s]

MarkovClustering MC-Diff-C mu=nan tau_ix=1 ..:   0%|          | 0/10 [00:00<?, ?it/s]

MarkovClustering MC-Diff-C mu=nan tau_ix=2 ..:   0%|          | 0/10 [00:00<?, ?it/s]

MarkovClustering MC-Diff-C mu=nan tau_ix=3 ..:   0%|          | 0/10 [00:00<?, ?it/s]

MarkovClustering MC-Diff-C mu=nan tau_ix=4 ..:   0%|          | 0/10 [00:00<?, ?it/s]

MarkovClustering MC-Diff-C mu=nan tau_ix=5 ..:   0%|          | 0/10 [00:00<?, ?it/s]

MarkovClustering MC-Diff-C mu=nan tau_ix=6 ..:   0%|          | 0/10 [00:00<?, ?it/s]

MarkovClustering MC-Diff-C mu=nan tau_ix=7 ..:   0%|          | 0/10 [00:00<?, ?it/s]

MarkovClustering MC-Diff-C mu=nan tau_ix=8 ..:   0%|          | 0/10 [00:00<?, ?it/s]

In [159]:
methods = ['louvain', 'label_propagation', 'leiden', 'infomap', 'sbm_dl', 'sbm_dl_nested']
# methods = ['louvain', 'leiden']
for method in methods:
    other_clustering = OtherClustering(paths=paths, input_graph=G, family='toy',
                                       ground_truth_clustering=ground_truth_clustering, method=method)
    other_clustering.run(num_trials=5, overwrite=True)

'louvain':   0%|          | 0/5 [00:00<?, ?it/s]

'label_propagation':   0%|          | 0/5 [00:00<?, ?it/s]

'leiden':   0%|          | 0/5 [00:00<?, ?it/s]

'infomap':   0%|          | 0/5 [00:00<?, ?it/s]

'sbm_dl':   0%|          | 0/5 [00:00<?, ?it/s]

'sbm_dl_nested':   0%|          | 0/5 [00:00<?, ?it/s]

## Plot clusterings

In [173]:
# read clustering info df 
test_df_whole = (
    pd.concat([
        pd.read_csv(cl_results_path),
        pd.read_csv(mc_results_path),
        pd.read_csv(other_results_path)
    ],
    ignore_index=True)
    .assign(method=lambda df_: df_.method.replace({'greedy_modularity': 'CNM',
                                                   'label_propagation': 'Label Prop',
                                                   'infomap': 'Infomap',
                                                   'eigenvector': 'eig',
                                                   'sbm_dl': 'SBM',
                                                   'sbm_dl_nested': 'Nested SBM'}))
)

test_df_whole.drop_duplicates(subset=['name', 'mu', 'avg_k', 'max_k', 'method', 'tau_ix'], inplace=True, keep='last')
display(f'{len(test_df_whole):,} rows', test_df_whole.method.unique(), test_df_whole.tail(10))

'144,270 rows'

array(['GW-C-exact', 'RW-D-exact', 'RW-C-exact', 'Diff-C-exact',
       'GW-D-exact', 'GW-D', 'CO-C', 'Diff-C', 'RW-C', 'GW-C', 'RW-D',
       'Diff-C-ER', 'RW-C-ER', 'CO-C-ER', 'RW-D-ER', 'MC-Diff-C',
       'MC-RW-C', 'Label Prop', 'Infomap', 'CNM', 'SBM', 'Nested SBM',
       'leiden', 'louvain'], dtype=object)

,timestamp,name,mu,avg_k,max_k,method,tau_max,tau_diff,tau_ix,tau,orig_clusters,running_time,metric,score,num_clusters,modularity,NMI,AMI,EC,output_cluster_path
147112,Fri Nov 11 11:36:24 2022,SBM-asymm,NaN,NaN,NaN,leiden,NaN,NaN,NaN,NaN,3,0.0017,modularity,0.37578,3,0.3758,0.9250,0.9234,0.9376,/N/slate/ssikdar/data-dump/corr-cluster/result...
147113,Fri Nov 11 11:36:25 2022,SBM-asymm,NaN,NaN,NaN,Infomap,NaN,NaN,NaN,NaN,3,0.0060,codelength,6.312158,2,0.2986,0.7337,0.7301,0.6667,/N/slate/ssikdar/data-dump/corr-cluster/result...
147114,Fri Nov 11 11:36:25 2022,SBM-asymm,NaN,NaN,NaN,SBM,NaN,NaN,NaN,NaN,3,0.0840,nmi,0.955482,3,0.3746,0.9555,0.9545,0.9678,/N/slate/ssikdar/data-dump/corr-cluster/result...
147115,Fri Nov 11 11:36:25 2022,SBM-asymm,NaN,NaN,NaN,Nested SBM,NaN,NaN,NaN,NaN,3,0.2468,nmi,0.910567,4,0.3458,0.9106,0.9076,0.9078,/N/slate/ssikdar/data-dump/corr-cluster/result...
147116,Fri Nov 11 11:53:25 2022,SBM-symm,NaN,NaN,NaN,louvain,NaN,NaN,NaN,NaN,3,0.0124,modularity,0.418559,3,0.4186,1.0000,1.0000,1.0000,/N/slate/ssikdar/data-dump/corr-cluster/result...
147117,Fri Nov 11 11:53:25 2022,SBM-symm,NaN,NaN,NaN,Label Prop,NaN,NaN,NaN,NaN,3,0.0029,nmi,0.0,1,0.0000,0.0000,0.0000,0.3333,/N/slate/ssikdar/data-dump/corr-cluster/result...
147118,Fri Nov 11 11:53:25 2022,SBM-symm,NaN,NaN,NaN,leiden,NaN,NaN,NaN,NaN,3,0.0021,modularity,0.418559,3,0.4186,1.0000,1.0000,1.0000,/N/slate/ssikdar/data-dump/corr-cluster/result...
147119,Fri Nov 11 11:53:26 2022,SBM-symm,NaN,NaN,NaN,Infomap,NaN,NaN,NaN,NaN,3,0.0051,codelength,6.140167,3,0.4186,1.0000,1.0000,1.0000,/N/slate/ssikdar/data-dump/corr-cluster/result...
147120,Fri Nov 11 11:53:26 2022,SBM-symm,NaN,NaN,NaN,SBM,NaN,NaN,NaN,NaN,3,0.1090,nmi,1.0,3,0.4186,1.0000,1.0000,1.0000,/N/slate/ssikdar/data-dump/corr-cluster/result...
147121,Fri Nov 11 11:53:26 2022,SBM-symm,NaN,NaN,NaN,Nested SBM,NaN,NaN,NaN,NaN,3,0.1939,nmi,1.0,3,0.4186,1.0000,1.0000,1.0000,/N/slate/ssikdar/data-dump/corr-cluster/result...


In [183]:
# do the two panel thingy 
def validate_both():
    acceptable_methods = set(test_df['method'].unique()) | set({'ground-truth'})
    return method_box_left.value in acceptable_methods and method_box_right.value in acceptable_methods
    
def validate_taus():
    tau_range = test_df.tau_ix.unique()
    return tau_ix_slider_left.value in tau_range and tau_ix_slider_right.value in tau_range 


def get_comm_labels_from_dataframe(which):
    methods_w_tau = ['MC', 'Gw', 'RW', 'Diff', 'CO']
    tau_str = ''
    if which == 'left':
        method = method_box_left.value
        tau_ix = tau_ix_slider_left.value
    else:
        method = method_box_right.value
        tau_ix = tau_ix_slider_right.value
    
    if method == 'ground-truth':
        comm_labels = ground_truth_comm_labels
        nmi, ami, ec = 1, 1, 1
    else:
        temp_df = test_df[test_df.method==method]
        if any(meth in method for meth in methods_w_tau):   # filter by tau ix for only these methdos
            tau_str = f'tau ix: {tau_ix}'
            temp_df = temp_df[temp_df.tau_ix==tau_ix]

        assert len(temp_df) > 0, f'Filtered DF empty for {method=}'
        nmi, ami, ec = temp_df.NMI.values[0], temp_df.AMI.values[0], temp_df.EC.values[0]
        comm_labels = read_file_into_dict(temp_df.output_cluster_path.values[0])
        
    num_clusters = len(set(comm_labels.values()))
    title = f'method: {method}    {tau_str}     clusters: {num_clusters:,}' #'<br>NMI: {nmi:.2f}    AMI: {ami:.2f}    EC: {ec:.2f}'
    return comm_labels, num_clusters, title 

def response_both(change):
    if validate_both() and validate_taus():
        comm_labels_left, num_clusters_left, title_left = get_comm_labels_from_dataframe('left')
        node_colors_left, node_texts_left = update_node_colors_and_texts(G=G, comm_labels=comm_labels_left)
        
        comm_labels_right, num_clusters_right, title_right = get_comm_labels_from_dataframe('right')
        node_colors_right, node_texts_right = update_node_colors_and_texts(G=G, comm_labels=comm_labels_right)
        
        with g.batch_update():
            g.update_traces(selector=dict(name='nodes'), marker=dict(color=node_colors_left), text=node_texts_left, row=1, col=1)
            g.update_traces(selector=dict(name='nodes'), marker=dict(color=node_colors_right), text=node_texts_right, row=1, col=2)
            
            g.layout.annotations[0].update(text=title_left)
            g.layout.annotations[1].update(text=title_right)
        
        return

In [162]:
(
    test_df
    .query('name==@name')
    .query('method=="Diff-C"')
    .query('tau_ix==0')
    # [['output_cluster_path']]
    # .values[0]
)

,timestamp,name,mu,avg_k,max_k,method,tau_max,tau_diff,tau_ix,tau,orig_clusters,running_time,metric,score,num_clusters,modularity,NMI,AMI,EC,output_cluster_path
120096,Fri Nov 11 11:53:13 2022,SBM-symm,NaN,NaN,NaN,Diff-C,0.0383,0.312016,0.0,2.220446e-16,3,0.0194,leakage_score,977.753425,3,0.4186,1.0,1.0,1.0,/N/slate/ssikdar/data-dump/corr-cluster/result...


In [184]:
# name = 'SBM-symm'

# sizes = [30, 30, 30]
# probs = [
#    [0.35, 0.05, 0.05],
#    [0.05, 0.35, 0.05],
#    [0.05, 0.05, 0.35]
#  ]


# G = nx.stochastic_block_model(sizes=sizes, p=probs, seed=1)
# G.name = name
# nx.set_node_attributes(G, name='pos', values=nx.spring_layout(G))

# ground_truth_comm_labels = nx.get_node_attributes(G, name='block')
# ground_truth_clustering = NodeClustering(dict_to_list_of_lists(ground_truth_comm_labels), graph=G)

test_df = test_df_whole.query('name==@name')

options = test_df.method.unique().tolist() + ['ground-truth']
method_box_left = widgets.Dropdown(
    description='Method (L)',
    value='RW-C',
    options=options,
)

method_box_right = widgets.Dropdown(
    description='Method (R):   ',
    value='RW-C',
    options=options,
)

tau_ix_slider_left = widgets.IntSlider(
    value=0,
    min=test_df.tau_ix.min(),
    max=test_df.tau_ix.max(),
    step=1,
    description='Tau index (L)',
    continuous_update=False,
    orientation='horizontal',
    readout=True
)

tau_ix_slider_right = widgets.IntSlider(
    value=0,
    min=test_df.tau_ix.min(),
    max=test_df.tau_ix.max(),
    step=1,
    description='Tau index (R)',
    continuous_update=False,
    orientation='horizontal',
    readout=True
)

mid_box = widgets.VBox(layout=widgets.Layout(width='20%'))
# mid_box = widgets.VBox()

left_box = widgets.VBox([method_box_left, tau_ix_slider_left])
right_box = widgets.VBox([method_box_right, tau_ix_slider_right])

container = widgets.HBox([left_box, mid_box, right_box])

traces = plot_graph(G, return_traces=True)

fig = make_subplots(
        rows=1, cols=2, subplot_titles=[' ', ' '],
        column_widths=[0.5, 0.5],
        vertical_spacing=0.0,)

fig.update_layout(title=dict(text=f'Clustering visualization:   {G.name}    |V| = {G.order():,}   |E| = {G.size():,}', y=0.98),
                  width=1500, height=700,
                  showlegend=False)

for col in [1, 2]:
    fig.update_xaxes(dict(showgrid=False, zeroline=False, showticklabels=False), row=1, col=col)
    fig.update_yaxes(dict(showgrid=False, zeroline=False, showticklabels=False), row=1, col=col)
    
g = go.FigureWidget(fig)
g.add_traces(traces, rows=1, cols=1)  # initialize both panels with the same graph
g.add_traces(traces, rows=1, cols=2);

method_box_left.observe(response_both, names="value")
method_box_right.observe(response_both, names="value")
tau_ix_slider_left.observe(response_both, names="value")
tau_ix_slider_right.observe(response_both, names="value")
# run this
widgets.VBox([container, g])

In [177]:
# name = 'SBM-asymm'
name = 'SBM-big-small-0.1'

# sizes = [30, 30, 30]
# probs = [
#    [0.35, 0.05, 0.05],
#    [0.05, 0.35, 0.1],
#    [0.05, 0.1, 0.35]
#  ]


# G = nx.stochastic_block_model(sizes=sizes, p=probs, seed=1)
# G.name = name
# nx.set_node_attributes(G, name='pos', values=nx.spring_layout(G))

# ground_truth_comm_labels = nx.get_node_attributes(G, name='block')
# ground_truth_clustering = NodeClustering(dict_to_list_of_lists(ground_truth_comm_labels), graph=G)

test_df = test_df_whole.query('name==@name')

options = test_df.method.unique().tolist() + ['ground-truth']
method_box_left = widgets.Dropdown(
    description='Method (L)',
    value='RW-C',
    options=options,
)

method_box_right = widgets.Dropdown(
    description='Method (R):   ',
    value='RW-C',
    options=options,
)

tau_ix_slider_left = widgets.IntSlider(
    value=0,
    min=test_df.tau_ix.min(),
    max=test_df.tau_ix.max(),
    step=1,
    description='Tau index (L)',
    continuous_update=False,
    orientation='horizontal',
    readout=True
)

tau_ix_slider_right = widgets.IntSlider(
    value=0,
    min=test_df.tau_ix.min(),
    max=test_df.tau_ix.max(),
    step=1,
    description='Tau index (R)',
    continuous_update=False,
    orientation='horizontal',
    readout=True
)

mid_box = widgets.VBox(layout=widgets.Layout(width='20%'))
# mid_box = widgets.VBox()

left_box = widgets.VBox([method_box_left, tau_ix_slider_left])
right_box = widgets.VBox([method_box_right, tau_ix_slider_right])

container = widgets.HBox([left_box, mid_box, right_box])

traces = plot_graph(G, return_traces=True)

fig = make_subplots(
        rows=1, cols=2, subplot_titles=[' ', ' '],
        column_widths=[0.5, 0.5],
        vertical_spacing=0.0,)

fig.update_layout(title=dict(text=f'Clustering visualization:   {G.name}    |V| = {G.order():,}   |E| = {G.size():,}', y=0.98),
                  width=1500, height=700,
                  showlegend=False)

for col in [1, 2]:
    fig.update_xaxes(dict(showgrid=False, zeroline=False, showticklabels=False), row=1, col=col)
    fig.update_yaxes(dict(showgrid=False, zeroline=False, showticklabels=False), row=1, col=col)
    
g = go.FigureWidget(fig)
g.add_traces(traces, rows=1, cols=1)  # initialize both panels with the same graph
g.add_traces(traces, rows=1, cols=2);

method_box_left.observe(response_both, names="value")
method_box_right.observe(response_both, names="value")
tau_ix_slider_left.observe(response_both, names="value")
tau_ix_slider_right.observe(response_both, names="value")
# run this
widgets.VBox([container, g])